In [1]:
import sqlite3
import pandas as pd

In [2]:
# 1. Create and connect to an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [3]:
# 2. Create the 'destinations' and 'attractions' tables
cursor.execute("""
CREATE TABLE destinations (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    country TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE attractions (
    id INTEGER PRIMARY KEY,
    destination_id INTEGER,
    name TEXT NOT NULL,
    type TEXT
);
""")

In [4]:
# 3. Insert sample data
# Note: Tokyo has multiple attractions, Paris has one, and Cairo has none (to test LEFT JOIN)
destinations_data = [
    (1, "Tokyo", "Japan"),
    (2, "Paris", "France"),
    (3, "Cairo", "Egypt")  
]

attractions_data = [
    (101, 1, "Shibuya Crossing", "Sightseeing"),
    (102, 1, "Tokyo Tower", "Landmark"),
    (103, 2, "Eiffel Tower", "Landmark")
]

cursor.executemany("INSERT INTO destinations VALUES (?, ?, ?);", destinations_data)
cursor.executemany("INSERT INTO attractions VALUES (?, ?, ?, ?);", attractions_data)
conn.commit()

# Helper function to print queries nicely using Pandas
def run_query(title, query):
    print(f"\n=== {title} ===")
    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))

# --- SQL OPERATION 1: INNER JOIN ---
# Shows only destinations that have matching attractions.
inner_join_query = """
SELECT 
    destinations.name AS Destination, 
    destinations.country AS Country, 
    attractions.name AS Attraction
FROM destinations
INNER JOIN attractions ON destinations.id = attractions.destination_id;
"""
run_query("INNER JOIN (Only destinations with matching attractions)", inner_join_query)

# --- SQL OPERATION 2: LEFT JOIN ---
# Includes every destination, even if an attraction is missing (like Cairo).
left_join_query = """
SELECT 
    destinations.name AS Destination, 
    destinations.country AS Country, 
    IFNULL(attractions.name, 'No Attractions Found') AS Attraction
FROM destinations
LEFT JOIN attractions ON destinations.id = attractions.destination_id;
"""
run_query("LEFT JOIN (Includes all destinations, even if missing attractions)", left_join_query)

# --- SQL OPERATION 3: CROSS JOIN ---
# Pairs every selected destination with every single attraction.
cross_join_query = """
SELECT 
    destinations.name AS Destination, 
    attractions.name AS Attraction
FROM destinations
CROSS JOIN attractions;
"""
run_query("CROSS JOIN (Pairs every destination with every attraction)", cross_join_query)

# --- SQL OPERATION 4: UNION ---
# Combines destination names and attraction names into a single list.
union_query = """
SELECT name AS 'Location/Place Name', 'Destination' AS Type FROM destinations
UNION
SELECT name AS 'Location/Place Name', 'Attraction' AS Type FROM attractions;
"""
run_query("UNION (Combined unique list of all names)", union_query)

# Close the database connection
conn.close()


=== INNER JOIN (Only destinations with matching attractions) ===
Destination Country       Attraction
      Tokyo   Japan Shibuya Crossing
      Tokyo   Japan      Tokyo Tower
      Paris  France     Eiffel Tower

=== LEFT JOIN (Includes all destinations, even if missing attractions) ===
Destination Country           Attraction
      Tokyo   Japan     Shibuya Crossing
      Tokyo   Japan          Tokyo Tower
      Paris  France         Eiffel Tower
      Cairo   Egypt No Attractions Found

=== CROSS JOIN (Pairs every destination with every attraction) ===
Destination       Attraction
      Tokyo Shibuya Crossing
      Tokyo      Tokyo Tower
      Tokyo     Eiffel Tower
      Paris Shibuya Crossing
      Paris      Tokyo Tower
      Paris     Eiffel Tower
      Cairo Shibuya Crossing
      Cairo      Tokyo Tower
      Cairo     Eiffel Tower

=== UNION (Combined unique list of all names) ===
Location/Place Name        Type
              Cairo Destination
       Eiffel Tower  Attraction
